# L-mode thermal confinement $\tau_{E,th}$ scaling
Reproduces the headline thermal confinement scaling of Kaye *et al.* 1997, Nucl. Fusion **37** 1303 (eq. 7, Table XIV) from the IMAS-migrated L-mode database.

In [1]:
import numpy as np

import _simdb_common as sc

db = sc.get_db()
sims = sc.query_dataset(db, "lmode")
print(f"{len(sims)} shots in lmode")


7233 shots in lmode


## Scaling variables

| Variable | IMAS path | Unit |
|---|---|---|
| $\tau_{E,th}$ (TAUTH) | `summary/global_quantities/tau_energy/value` | s |
| $I_p$ | `summary/global_quantities/ip/value` | A |
| $B_T$ | `summary/global_quantities/b0/value` | T |
| $\kappa$ | `summary/boundary/elongation/value` | - |
| $R_{geo}$ | `summary/boundary/geometric_axis_r/value` | m |
| $a$ | `summary/boundary/minor_radius/value` | m |
| $\bar n_e$ | `summary/line_average/n_e/value` | m^-3 |
| $M_{eff}$ | `summary/line_average/meff_hydrogenic/value` | AMU |
| $P_{l,th}$ (PLTH) | `summary/global_quantities/power_loss/value` | W |
| PHASE | `temporary` (`PHASE`) | - |
| SELDB1 | `temporary` (`SELDB1`) | - |
| TOK | `summary/machine` | - |

The standard subset of the paper (PHASE = L, hydrogenic plasma and beam, TAUTH present) is flagged by `SELDB1` = 11.

In [2]:
rows = []
for sim in sims:
    md = sim.meta_dict()
    phase = sc.temp_str(md, "PHASE", n=0)
    n = len(phase)
    if n == 0:
        continue
    rows.append((
        np.full(n, md.get("machine", "")),
        phase,
        sc.temp(md, "SELDB1", n=n),
        sc.path(md, "global_quantities", "tau_energy", "value", n=n),
        np.abs(sc.path(md, "global_quantities", "ip", "value", n=n)),
        np.abs(sc.path(md, "global_quantities", "b0", "value", n=n)),
        sc.path(md, "boundary", "elongation", "value", n=n),
        sc.path(md, "boundary", "geometric_axis_r", "value", n=n),
        sc.path(md, "boundary", "minor_radius", "value", n=n),
        sc.path(md, "line_average", "n_e", "value", n=n),
        sc.path(md, "line_average", "meff_hydrogenic", "value", n=n),
        sc.path(md, "global_quantities", "power_loss", "value", n=n),
    ))

TOK, PHASE, SELDB1, TAU, IP, BT, KAPPA, RGEO, AMIN, NEL, MEFF, PLTH = (
    np.concatenate(c) for c in zip(*rows))
print(f"{len(TAU)} timeslices from {len(np.unique(TOK))} machines")


9506 timeslices from 22 machines


In [3]:
# Paper units: MA, T, m, 1e19 m^-3, AMU, MW
ip_ma    = IP / 1e6
ne_19    = NEL / 1e19
Ploss_MW = PLTH / 1e6
inv_eps  = RGEO / AMIN

if np.any(np.isfinite(SELDB1)):
    sel = SELDB1 == 11
else:
    print("SELDB1 absent, falling back to the Appendix B definition")
    sel = (PHASE.astype(str) == "L") & np.isfinite(TAU)

regressors = {"Ip": ip_ma, "Bt": BT, "kappa": KAPPA, "R": RGEO,
              "R/a": inv_eps, "ne": ne_19, "Meff": MEFF, "P": Ploss_MW}
ok = np.isfinite(TAU) & (TAU > 0)
for x in regressors.values():
    ok &= np.isfinite(x) & (x > 0)

sel = sel & ok
print(f"N = {sel.sum()} (paper N = 1312)")


N = 1312 (paper N = 1312)


## Headline scaling
Kaye *et al.* eq. (7), the thermal confinement scaling of the standard L-mode subset:

$$\tau_{E,th} = 0.023\, I_p^{0.96}\, B_T^{0.03}\, \kappa^{0.64}\, R^{1.83}\, (R/a)^{0.06}\, \bar n_e^{0.40}\, M_{eff}^{0.20}\, P^{-0.73}$$

in seconds, megamps, teslas, metres, $10^{19}$ $\text{m}^{-3}$, atomic mass units and megawatts. Fit done using OLS on log(data)

In [4]:
y = np.log(TAU[sel])
X = np.column_stack([np.ones_like(y)] + [np.log(x[sel]) for x in regressors.values()])

coef, *_ = np.linalg.lstsq(X, y, rcond=None)
resid = y - X @ coef
n, p = X.shape
cov = (resid @ resid / (n - p)) * np.linalg.pinv(X.T @ X)
se = np.sqrt(np.diag(cov))

# Paper Table XIV, "Estimate" and "Standard error" columns
paper = {"const": (0.023, 0.001), "Ip": (0.96, 0.02), "Bt": (0.03, 0.02),
         "kappa": (0.64, 0.03), "R": (1.83, 0.03), "R/a": (0.06, 0.04),
         "ne": (0.40, 0.02), "Meff": (0.20, 0.02), "P": (-0.73, 0.01)}

print(f"{'param':8s} {'this fit':>18s} {'paper (Tbl XIV)':>18s}")
a0 = np.exp(coef[0])
print(f"{'const':8s} {a0:>8.4f} +- {a0 * se[0]:<6.4f} {paper['const'][0]:>8.3f} +- {paper['const'][1]:<6.3f}")
for name, c, s in zip(regressors, coef[1:], se[1:]):
    pc, ps = paper[name]
    print(f"{name:8s} {c:>8.3f} +- {s:<6.3f} {pc:>8.2f} +- {ps:<6.2f}")

rmse = np.sqrt(np.mean(resid**2))
r2 = 1.0 - resid @ resid / np.sum((y - y.mean())**2)
print(f"\nN = {n}   RMSE(log) = {rmse:.3f}   R^2 = {r2:.3f}   (paper N = 1312, RMSE = 15.8%, R^2 = 0.97)")


param              this fit    paper (Tbl XIV)
const      0.0230 +- 0.0012    0.023 +- 0.001 
Ip          0.966 +- 0.021      0.96 +- 0.02  
Bt          0.043 +- 0.019      0.03 +- 0.02  
kappa       0.639 +- 0.033      0.64 +- 0.03  
R           1.800 +- 0.034      1.83 +- 0.03  
R/a         0.058 +- 0.042      0.06 +- 0.04  
ne          0.405 +- 0.018      0.40 +- 0.02  
Meff        0.200 +- 0.019      0.20 +- 0.02  
P          -0.735 +- 0.009     -0.73 +- 0.01  

N = 1312   RMSE(log) = 0.159   R^2 = 0.974   (paper N = 1312, RMSE = 15.8%, R^2 = 0.97)


## Cross-check against the raw CSV

In [5]:
import csv

raw = [r for r in csv.DictReader(open("../resources/input/l_mode_data.csv")) if r["SELDB1"] == "11"]
rawcol = lambda name: np.array([float(r[name]) for r in raw])

raw_reg = {"Ip": np.abs(rawcol("IP")) / 1e6, "Bt": np.abs(rawcol("BT")), "kappa": rawcol("KAPPA"),
           "R": rawcol("RGEO"), "R/a": rawcol("RGEO") / rawcol("AMIN"), "ne": rawcol("NEL") / 1e19,
           "Meff": rawcol("MEFF"), "P": rawcol("PLTH") / 1e6}

y_raw = np.log(rawcol("TAUTH"))
X_raw = np.column_stack([np.ones_like(y_raw)] + [np.log(x) for x in raw_reg.values()])
coef_raw, *_ = np.linalg.lstsq(X_raw, y_raw, rcond=None)

print(f"{'param':8s} {'migrated':>10s} {'raw CSV':>10s} {'paper':>8s} {'mig - csv':>10s}")
print(f"{'const':8s} {np.exp(coef[0]):>10.4f} {np.exp(coef_raw[0]):>10.4f} {paper['const'][0]:>8.3f} "
      f"{np.exp(coef[0]) - np.exp(coef_raw[0]):>+10.4f}")
for name, c, c_raw in zip(regressors, coef[1:], coef_raw[1:]):
    print(f"{name:8s} {c:>10.3f} {c_raw:>10.3f} {paper[name][0]:>8.2f} {c - c_raw:>+10.3f}")
print(f"\nN = {sel.sum()} migrated, {len(y_raw)} raw CSV")


param      migrated    raw CSV    paper  mig - csv
const        0.0230     0.0230    0.023    +0.0000
Ip            0.966      0.966     0.96     +0.000
Bt            0.043      0.043     0.03     +0.000
kappa         0.639      0.639     0.64     +0.000
R             1.800      1.800     1.83     +0.000
R/a           0.058      0.058     0.06     +0.000
ne            0.405      0.405     0.40     +0.000
Meff          0.200      0.200     0.20     +0.000
P            -0.735     -0.735    -0.73     +0.000

N = 1312 migrated, 1312 raw CSV
